**Import Libraries**

In [58]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

print("Libraries imported!")

Libraries imported!


Feature Engineering Strategy

**Raw landmarks alone aren't enough!** We need features that are:
 
 1. **Translation-invariant** → Works regardless of hand position
 2. **Scale-invariant** → Works for hands at different distances
 3. **Rotation-invariant** → Works at any hand angle
 
Our Feature Set:
 
 1. **Normalized coordinates** (relative to wrist)
 2. **Finger distances** (tip to wrist)
 3. **Finger angles** (bend angles)
 4. **Finger states** (extended vs folded)

**Feature Extraction Functions**

In [62]:
class HandFeatureExtractor:
    
    def __init__(self):
        # Landmark indices for each finger tip
        self.finger_tips = [4, 8, 12, 16, 20]  
        self.finger_pips = [2, 6, 10, 14, 18]  
        self.finger_mcps = [1, 5, 9, 13, 17]   

In [64]:
 def extract_features(self, hand_landmarks) -> np.ndarray:
       
        # Extract landmark coordinates
        landmarks = self._get_landmark_array(hand_landmarks)
        
        # 1. Normalize coordinates (relative to wrist)
        normalized = self._normalize_landmarks(landmarks)
        
        # 2. Calculate distances from fingertips to wrist
        distances = self._calculate_distances(landmarks)
        
        # 3. Calculate finger angles
        angles = self._calculate_angles(landmarks)
        
        # 4. Finger extension states
        finger_states = self._get_finger_states(landmarks)
        
        # Combine all features
        features = np.concatenate([
            normalized.flatten(),  
            distances,             
            angles,               
            finger_states
        ])
        
        return features

In [66]:
  def _get_landmark_array(self, hand_landmarks) -> np.ndarray:
        # Convert landmarks to numpy array
        landmarks = []
        for lm in hand_landmarks.landmark:
            landmarks.append([lm.x, lm.y])
        return np.array(landmarks)

In [68]:
 def _normalize_landmarks(self, landmarks: np.ndarray) -> np.ndarray:
        # Normalize landmarks relative to wrist (landmark 0)
        wrist = landmarks[0]
        normalized = landmarks - wrist
        
        # Scale to unit size (based on hand size)
        hand_size = np.max(np.linalg.norm(normalized, axis=1))
        if hand_size > 0:
            normalized = normalized / hand_size
        
        return normalized

In [70]:
def _calculate_distances(self, landmarks: np.ndarray) -> np.ndarray:
      # Calculate Euclidean distances from fingertips to wrist
        wrist = landmarks[0]
        distances = []
        
        for tip_idx in self.finger_tips:
            tip = landmarks[tip_idx]
            dist = np.linalg.norm(tip - wrist)
            distances.append(dist)
        
        return np.array(distances)

In [72]:
def _calculate_angles(self, landmarks: np.ndarray) -> np.ndarray:
       # Calculate finger bend angles
        angles = []
        
        for tip_idx, pip_idx, mcp_idx in zip(
            self.finger_tips, self.finger_pips, self.finger_mcps
        ):
            # Vectors
            v1 = landmarks[pip_idx] - landmarks[mcp_idx]
            v2 = landmarks[tip_idx] - landmarks[pip_idx]
            
            # Calculate angle
            angle = self._angle_between_vectors(v1, v2)
            angles.append(angle)
        
        return np.array(angles)

In [74]:
 def _angle_between_vectors(self, v1: np.ndarray, v2: np.ndarray) -> float:
        # Calculate angle between two vectors in degrees
        cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6)
        cos_angle = np.clip(cos_angle, -1.0, 1.0)
        angle = np.arccos(cos_angle)
        return np.degrees(angle)

In [76]:
def _get_finger_states(self, landmarks: np.ndarray) -> np.ndarray:
        # Determine if fingers are extended (1) or folded (0)
        states = []
        wrist = landmarks[0]
        
        for tip_idx, pip_idx in zip(self.finger_tips, self.finger_pips):
            tip = landmarks[tip_idx]
            pip = landmarks[pip_idx]
            
            # Finger is extended if tip is farther from wrist than PIP joint
            tip_dist = np.linalg.norm(tip - wrist)
            pip_dist = np.linalg.norm(pip - wrist)
            
            is_extended = 1 if tip_dist > pip_dist * 1.1 else 0
            states.append(is_extended)
        
        return np.array(states)

In [78]:
# Initialize extractor
feature_extractor = HandFeatureExtractor()
print("Feature extractor initialized!")

Feature extractor initialized!


**Test Feature Extraction**

In [81]:
def test_feature_extraction():
   # Test feature extraction on live webcam
    cap = cv2.VideoCapture(0)
    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5
    )
    
    print("Show your hand to extract features")
    
    features_extracted = False
    
    while cap.isOpened() and not features_extracted:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)
        
        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            
            # Extract features
            features = feature_extractor.extract_features(hand_landmarks)
            
            # Draw landmarks
            mp_drawing.draw_landmarks(
                frame, hand_landmarks, mp_hands.HAND_CONNECTIONS
            )
            
            # Display feature info
            cv2.putText(frame, f"Features: {len(features)}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            print(f"\n Extracted {len(features)} features!")
            print(f"Feature vector shape: {features.shape}")
            print(f"Sample features: {features[:10]}")
            
            features_extracted = True
        
        cv2.imshow('Feature Extraction Test', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    hands.close()

**Visualize Features**

In [85]:
def visualize_finger_states(hand_landmarks):
    # Visualize which fingers are extended
    landmarks = feature_extractor._get_landmark_array(hand_landmarks)
    finger_states = feature_extractor._get_finger_states(landmarks)
    
    finger_names = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
    states_str = ['Folded', 'Extended']
    
    plt.figure(figsize=(10, 4))
    colors = ['red' if s == 0 else 'green' for s in finger_states]
    plt.bar(finger_names, finger_states, color=colors, alpha=0.7)
    plt.ylim(0, 1.5)
    plt.ylabel('State')
    plt.title('Finger Extension States')
    plt.yticks([0, 1], states_str)
    
    for i, (name, state) in enumerate(zip(finger_names, finger_states)):
        plt.text(i, state + 0.1, states_str[int(state)], 
                ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

 **Feature Summary**

Our 57-dimensional feature vector consists of:

1. **Normalized (x, y) coordinates** → 42 features (21 landmarks × 2)
2. **Fingertip-to-wrist distances** → 5 features
3. **Finger bend angles** → 5 features
4. **Finger extension states** → 5 features (binary)